# M4 · App

Lanza la app de chat (Gradio) en la que el usuario escribe un caso clínico o adjunta la foto de
una nota clínica, y recibe los hallazgos clínicos detectados y los fragmentos de las guías que
los respaldan, con una respuesta redactada por el LLM.

Flujo: imagen → VLM u OCR (M4/vision) → texto → NER (M1/M2) → por cada hallazgo:
orquestación y retrieval (M3) → fragmentos de guía → generación con Groq (M3).

**Antes de empezar:** Entorno de ejecución > Cambiar tipo de entorno de ejecución > **T4 GPU**.
Sin GPU, la app lee las imágenes con OCR en lugar del VLM.

**Requisitos:**
- La clave `GROQ_API_KEY` en los Secretos de Colab (ícono de llave), con acceso al notebook activado.
- En Drive, dentro de `MyDrive/TopicosIA/Proyecto-Salud`: el adaptador LoRA del NER y los chunks y el manifiesto del corpus de M3.

## 1. Clonar el repositorio e instalar dependencias

Clona la rama del repo con el código de la app y entra a la carpeta. Luego instala las
dependencias de `M4/app/requirements.txt`, que reúne las del NER (M2), la visión (M4), el
retrieval y la generación (M3) y Gradio para la interfaz.

La instalación tarda unos minutos.

In [ ]:
!git clone --branch dev --single-branch https://github.com/luisNP21/Topicos-IA.git /content/corpus
%cd /content/corpus
!pip install -q -r M4/app/requirements.txt

## 2. Montar Drive, configurar el entorno y verificar recursos

- Monta Google Drive, donde están los datos que no se versionan en git.
- Lee la clave de Groq desde los Secretos de Colab y escribe un archivo `.env` con
  `PROJECT_ROOT` (la carpeta del proyecto en Drive) y `GROQ_API_KEY`.
- Verifica que haya GPU y que existan los tres recursos que necesita la app:
  - el adaptador LoRA del NER (`M1/saved_models/clinical_bert-distemist-lora`);
  - los chunks del corpus de guías (`M3/data/guias_clinicas/chunks`);
  - el manifiesto del corpus (`M3/data/guias_clinicas/corpus_manifest.json`).

Antes de continuar, todo debe salir `OK` y `GPU: True`.

In [ ]:
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")

PROJECT_ROOT = "/content/drive/MyDrive/TopicosIA/Proyecto-Salud"
GROQ_API_KEY = userdata.get("GROQ_API_KEY")   # Secretos de Colab

with open("/content/corpus/.env", "w") as f:
    f.write(f"PROJECT_ROOT={PROJECT_ROOT}\n")
    f.write(f"GROQ_API_KEY={GROQ_API_KEY}\n")

import torch
print("GPU:", torch.cuda.is_available())
print("Clave de Groq cargada:", bool(GROQ_API_KEY))
for p in ["M1/saved_models/clinical_bert-distemist-lora/adapter_config.json",
          "M3/data/guias_clinicas/chunks",
          "M3/data/guias_clinicas/corpus_manifest.json"]:
    print(f"{'OK   ' if (Path(PROJECT_ROOT) / p).exists() else 'FALTA'} {p}")

## 3. Lanzar la app

Arranca la app con un enlace público temporal (`--share`).

La primera vez tarda unos minutos: descarga los modelos (embeddings, reranker, Clinical BERT)
y reconstruye el índice de las guías. Cuando esté lista aparece una línea con
`Running on public URL: https://....gradio.live`; ese es el enlace para abrir la app.

- La app funciona mientras esta celda siga corriendo. Si se detiene o se desconecta la sesión,
  el enlace deja de servir y hay que volver a ejecutarla.
- La primera imagen tarda más, porque es cuando se carga el VLM (alrededor de un minuto por página en una T4).
- Es una demostración académica: no se deben ingresar datos reales de pacientes.

In [ ]:
!python M4/app/app.py --share